# 07 · Síntesis: las tres representaciones, lado a lado

Cierre de la sesión: la misma frase del corpus de trabajo, vectorizada con
los tres enfoques vistos, y los criterios para elegir uno u otro fuera del
salón de clase.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

corpus = ["el gato duerme en el sofá", "el perro duerme en la alfombra", "el sismo sacudió la costa"]
frase = corpus[0]

bow = CountVectorizer(token_pattern=r"(?u)\b\w+\b").fit(corpus)
tfidf = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b").fit(corpus)
sbert = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

vector_bow = bow.transform([frase]).toarray()[0]
vector_tfidf = tfidf.transform([frase]).toarray()[0]
vector_embedding = sbert.encode([frase])[0]

for nombre, vector in [("Bag-of-Words", vector_bow), ("TF-IDF", vector_tfidf), ("Embedding", vector_embedding)]:
    ceros = int((vector == 0).sum())
    print(f"{nombre:14} dim={len(vector):>4}   ceros={ceros:>4} ({ceros/len(vector):.0%})   primeros valores: {np.round(vector[:6], 2)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Bag-of-Words   dim=  11   ceros=   6 (55%)   primeros valores: [0 0 1 2 1 1]
TF-IDF         dim=  11   ceros=   6 (55%)   primeros valores: [0.   0.   0.36 0.55 0.36 0.47]
Embedding      dim= 384   ceros=   0 (0%)   primeros valores: [ 0.48 -0.29 -0.13  0.18  0.07  0.3 ]


| Representación | Dimensiones | Ceros | Qué determina los valores |
|---|---|---|---|
| Bag-of-Words | tamaño del vocabulario | casi todos | el conteo: cuántas veces aparece cada palabra |
| TF-IDF | tamaño del vocabulario | casi todos | el conteo, pesado por la rareza en el corpus |
| Embedding | fija (384 aquí; 300-1536 típico) | ninguno | un modelo entrenado sobre millones de oraciones |

Registra apariciones sin ninguna noción de significado (BoW); distingue lo
informativo de lo común, pero tampoco codifica significado (TF-IDF);
codifica significado y contexto, a costa de no ser interpretable
(embedding). Un mismo texto, tres formas de convertirlo en números — la
diferencia está en **qué determina esos valores**: el conteo, la
estadística del corpus, o el modelo.

## Criterios de selección

```
¿El significado depende del contexto?
├── NO (vocabulario cerrado, coincidencias exactas importan)
│     └── TF-IDF        → códigos, nombres propios, normativa;
│                          sin GPU, explicable término a término
└── SÍ (polisemia, paráfrasis)
      ├── poco cómputo disponible
      │     └── Word2Vec  → semántica barata, o como entrada de otro modelo
      └── GPU e inferencia disponibles
            └── SBERT / Transformers → búsqueda semántica, RAG,
                                        clasificación fina
```

En producción casi nunca se elige uno solo: los buscadores combinan ambos
enfoques (**búsqueda híbrida**) — primero recuperan candidatos con los dos
métodos y luego reordenan solo los mejores con un modelo más caro y
preciso. Así funcionó el buscador de `05_transformers-sbert.ipynb`.

## Errores frecuentes en implementación

1. **Usar embeddings para todo.** Para buscar un código de factura, TF-IDF
   gana y cuesta mil veces menos cómputo.
2. **Promediar vectores sin pensar.** El promedio de un historial diverso
   —o dominado por una compra repetida, como se vio en
   `06_sistema-recomendacion.ipynb`— cae en un punto que no representa a
   nadie.
3. **Comparar cosenos entre modelos distintos.** 0.8 en un modelo y 0.8 en
   otro no significan lo mismo: los espacios vectoriales no son
   comparables entre sí.
4. **Olvidar el idioma del preentrenamiento.** Un modelo entrenado en
   inglés rinde mal en español técnico, aunque "funcione" en apariencia.
5. **No fijar una línea base.** Sin medir TF-IDF primero, no se sabe si el
   modelo caro aporta algo de verdad.

## Glosario

| Término | Definición |
|---|---|
| Vector disperso | Lista larga, casi toda de ceros. Bag-of-Words, TF-IDF. |
| Vector denso / embedding | Lista corta sin ceros, aprendida. Word2Vec, BERT. |
| Similitud coseno | Ángulo entre dos vectores. 1 = iguales, 0 = sin relación. |
| IDF | Dial que baja el peso de las palabras que salen en todos lados. |
| Negative sampling | Aproximación que evita recorrer todo el vocabulario en cada paso de Word2Vec. |
| Self-attention | Cada palabra se recalcula mezclando a las demás según relevancia. |
| Contextual | El vector cambia según la frase. "Banco" ya no es uno solo. |
| Bi-encoder | Vectorizar por separado e indexar. Lo que hace viable buscar (SBERT). |

## Referencias

- Mikolov et al. (2013) — Word2Vec: la idea original.
- Levy & Goldberg (2014) — Word2Vec como factorización matricial.
- Pennington et al. (2014) — GloVe.
- Vaswani et al. (2017) — *Attention Is All You Need*.
- Devlin et al. (2019) — BERT.
- Reimers & Gurevych (2019) — Sentence-BERT.
- Jurafsky & Martin — *Speech and Language Processing*, cap. 6 (libre en línea).

**Idea central de la sesión:** vectorizar consiste en decidir qué
relaciones queda en condiciones de detectar el modelo.